# Preparando Dados

## Carregando e Lendo Dataset

In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files: /kaggle/input/fashionmnist


In [ ]:
print(os.listdir(path))

['t10k-labels-idx1-ubyte', 't10k-images-idx3-ubyte', 'fashion-mnist_test.csv', 'fashion-mnist_train.csv', 'train-labels-idx1-ubyte', 'train-images-idx3-ubyte']


## Separando Treino e teste

In [ ]:
import pandas as pd
train_df = pd.read_csv(os.path.join(path, 'fashion-mnist_train.csv'))
test_df = pd.read_csv(os.path.join(path, 'fashion-mnist_test.csv'))

print("treino: ", train_df.shape)
print("teste: ", test_df.shape)

treino:  (60000, 785)
teste:  (10000, 785)


In [ ]:
X_train = train_df.drop('label', axis=1)
y_train = train_df['label']

X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

In [ ]:
print(X_train.shape)  # (60000, 784)
print(y_train.shape)  # (60000,)

(60000, 784)
(60000,)


In [ ]:
#Normalização
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_mn = scaler.fit_transform(X_train)
X_test_mn = scaler.transform(X_test)

#Base Line

Decision Tree Classifier utilizando todas as Features

In [33]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [34]:
clf = DecisionTreeClassifier(random_state=42)

clf.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [35]:
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.7963
['pixel1' 'pixel2' 'pixel3' 'pixel4' 'pixel5' 'pixel6' 'pixel7' 'pixel8'
 'pixel9' 'pixel10' 'pixel11' 'pixel12' 'pixel13' 'pixel14' 'pixel15'
 'pixel16' 'pixel17' 'pixel18' 'pixel19' 'pixel20' 'pixel21' 'pixel22'
 'pixel23' 'pixel24' 'pixel25' 'pixel26' 'pixel27' 'pixel28' 'pixel29'
 'pixel30' 'pixel31' 'pixel32' 'pixel33' 'pixel34' 'pixel35' 'pixel36'
 'pixel37' 'pixel38' 'pixel39' 'pixel40' 'pixel41' 'pixel42' 'pixel43'
 'pixel44' 'pixel45' 'pixel46' 'pixel47' 'pixel48' 'pixel49' 'pixel50'
 'pixel51' 'pixel52' 'pixel53' 'pixel54' 'pixel55' 'pixel56' 'pixel57'
 'pixel58' 'pixel59' 'pixel60' 'pixel61' 'pixel62' 'pixel63' 'pixel64'
 'pixel65' 'pixel66' 'pixel67' 'pixel68' 'pixel69' 'pixel70' 'pixel71'
 'pixel72' 'pixel73' 'pixel74' 'pixel75' 'pixel76' 'pixel77' 'pixel78'
 'pixel79' 'pixel80' 'pixel81' 'pixel82' 'pixel83' 'pixel84' 'pixel85'
 'pixel86' 'pixel87' 'pixel88' 'pixel89' 'pixel90' 'pixel91' 'pixel92'
 'pixel93' 'pixel94' 'pixel95' 'pixel96' 'pixel97' 'pixel98

Acurácia no conjunto de teste: 0.7943

Porcentagem de features selecionadas (em relação aos dados originais): 100%

Tempo de treinamento: 1min

Tempo necessário para encontrar o subconjunto de features: N/A (usaram-se todas)


# Wrapper
Forward Selection

In [ ]:
# Tirando uma amostra do dataset



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SequentialFeatureSelector


# Modelo base
dt = DecisionTreeClassifier(random_state=42)

# Forward Selection
sfs = SequentialFeatureSelector(
    dt,
    n_features_to_select='auto',  # ou um número específico
    direction='forward',
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

# IMPORTANTE: apenas dados de treinamento
sfs.fit(X_train, y_train)

# Features selecionadas
selected_features = X_train.columns[sfs.get_support()]

print(f"Quantidade de features selecionadas: {len(selected_features)}")



'\n# Forward Selection\nsfs = SequentialFeatureSelector(\n    dt,\n    n_features_to_select=\'auto\',  # ou um número específico\n    direction=\'forward\',\n    scoring=\'accuracy\',\n    cv=5,\n    n_jobs=-1\n)\n\n# IMPORTANTE: apenas dados de treinamento\nsfs.fit(X_train, y_train)\n\n# Features selecionadas\nselected_features = X_train.columns[sfs.get_support()]\n\nprint(f"Quantidade de features selecionadas: {len(selected_features)}")\n'

In [ ]:

X_train_fs = sfs.transform(X_train)
X_test_fs = sfs.transform(X_test)


'\nX_train_fs = sfs.transform(X_train)\nX_test_fs = sfs.transform(X_test)\n'

In [ ]:
n_original = X_train.shape[1]
n_selected = len(selected_features)

percentual = (n_selected / n_original) * 100

print(f"Features originais: {n_original}")
print(f"Features selecionadas: {n_selected}")
print(f"Percentual utilizado: {percentual:.2f}%")

NameError: name 'selected_features' is not defined

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

clf.fit(X_train_fs, y_train)

pred = clf.predict(X_test_fs)

NameError: name 'X_train_fs' is not defined

# Genetic Algorithm

In [ ]:
!pip install sklearn-genetic-opt

In [ ]:
import time
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn_genetic import GAFeatureSelectionCV

In [24]:
clf = DecisionTreeClassifier(random_state=42)

ga_estimator = GAFeatureSelectionCV(
    estimator=clf,
    cv=3,
    scoring="accuracy",
    population_size=10,
    generations=10,
    n_jobs=-1,
    verbose=True,
    keep_top_k=3,
    elitism=True,
)

timer_fs = time.process_time()
ga_estimator.fit(X_train, y_train)
timer_fe = time.process_time()

features = ga_estimator.support_

timer_ps = time.process_time()
y_predict_ga = ga_estimator.predict(X_test)
timer_pe = time.process_time()

accuracy = accuracy_score(y_test, y_predict_ga)

print('Acurácia final: \n' + str(accuracy))
print('Quantidade de features selecionadas: \n' + str(len(features)))
print('Tempo de Treinamento: \n' + str( (timer_pe - timer_ps) ) + ' segundos.')
print('Tempo de Escolha de features: \n' + str( (timer_fe - timer_fs) ) + ' segundos.')

/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


gen	nevals	fitness	fitness_std	fitness_max	fitness_min
0  	10    	0.78608	0.00140951 	0.78905    	0.78485    
1  	20    	0.787698	0.00123824 	0.7891     	0.785667   
2  	20    	0.788168	0.00101153 	0.789867   	0.7865     
3  	20    	0.788252	0.00104657 	0.789333   	0.785783   
4  	20    	0.788403	0.000665824	0.789333   	0.787483   
5  	20    	0.78808 	0.00116866 	0.789333   	0.786133   
6  	20    	0.788648	0.000853179	0.7899     	0.7876     
7  	20    	0.788865	0.00112236 	0.79       	0.786533   
8  	20    	0.789005	0.00095708 	0.79       	0.787433   
9  	20    	0.789052	0.00107097 	0.790167   	0.78675    
10 	20    	0.788578	0.00128731 	0.790167   	0.78675    


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but GAFeatureSelectionCV was fitted without feature names
  warnings.warn(


Acurácia final: 
0.7982
Quantidade de features selecionadas: 
784
Tempo de Treinamento: 
0.2984305609999751 segundos.
Tempo de Escolha de features: 
124.97259982900002 segundos.





Resultados do primeiro teste da bib: (Demorou aprox. 40 min com pop.size=10 e gen=10)
```
gen	nevals	fitness 	fitness_std	fitness_max	fitness_min
0  	10    	0.786125	0.00195769 	0.788717   	0.78145    
1  	20    	0.787605	0.00136043 	0.790217   	0.786067   
2  	20    	0.788953	0.0010432  	0.790217   	0.78715    
3  	20    	0.789122	0.00126724 	0.790783   	0.786083   
4  	20    	0.78914 	0.000991217	0.79045    	0.78755    
5  	20    	0.789102	0.00175186 	0.79045    	0.784817   
6  	20    	0.78951 	0.00119292 	0.79045    	0.78725    
7  	20    	0.78917 	0.00133807 	0.79045    	0.78735    
8  	20    	0.789203	0.00133101 	0.79045    	0.786917   
9  	20    	0.789738	0.00107858 	0.79045    	0.787467
```

Outro teste da bib:



```
gen	nevals	fitness	fitness_std	fitness_max	fitness_min
0  	10    	0.78608	0.00140951 	0.78905    	0.78485    
1  	20    	0.787698	0.00123824 	0.7891     	0.785667   
2  	20    	0.788168	0.00101153 	0.789867   	0.7865     
3  	20    	0.788252	0.00104657 	0.789333   	0.785783   
4  	20    	0.788403	0.000665824	0.789333   	0.787483   
5  	20    	0.78808 	0.00116866 	0.789333   	0.786133   
6  	20    	0.788648	0.000853179	0.7899     	0.7876     
7  	20    	0.788865	0.00112236 	0.79       	0.786533   
8  	20    	0.789005	0.00095708 	0.79       	0.787433   
9  	20    	0.789052	0.00107097 	0.790167   	0.78675    
10 	20    	0.788578	0.00128731 	0.790167   	0.78675

Acurácia final:
0.7982
Quantidade de features selecionadas:
784
Tempo de Treinamento:
0.2984305609999751 segundos.
Tempo de Escolha de features:
124.97259982900002 segundos.
```



## Genetic Algorithm

In [25]:
import time

from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score

## Ferramentas

In [30]:
# Calcular a acurácia antes de chamar
def fitness(clf: DecisionTreeClassifier, acc: float):
  """
  clf: Classificador a ser avaliado
  acc: Acurácia

  Calcular a acurácia antes de chamar!
  """
  f = clf.n_features_in_
  return ( (acc * 1000) - (f * 1.5))

In [27]:
def crossover(parentA: DecisionTreeClassifier, parentB: DecisionTreeClassifier, debug: bool):
  offspring = DecisionTreeClassifier(random_state=42)
  offspring.n_features_in_ = (parentA.n_features_in_ + parentB.n_features_in_) / 2

  if (debug):
    print("---- CROSSOVER ----")
    print('Qtde. feat. filho: ', str(offspring.n_features_in_))

  return offspring

In [28]:
def gen(pop: int, X_train, y_train):
  candidates = []
  for i in range(pop):
    clf = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
    candidates.append(clf)
  return candidates

## Algoritmo

## Execução